In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle, \
    draw_fusion_keyframe_traj


In [3]:
image_dir = Path("/home/lty/datasets/RealUAV/city2/")
seu_uav_dir = image_dir / "uav"
seu_tif_dir = image_dir / "tif"
output_dir = Path("/home/lty/outputs/RealUAV/city2")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc.txt"# 保存定位结果

In [4]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [4]:
import time
t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/05/11 14:20:32 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2025/05/11 14:20:32 hloc INFO] Skipping the extraction.
[2025/05/11 14:20:32 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
[2025/05/11 14:20:32 hloc INFO] Skipping the matching.


Feature extraction time: 0.114s
Feature matching time: 0.006s


In [5]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/datasets/RealUAV/city2/real_uav_city2_geotransform.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [12122606.937915787, 0.2985821417389691, 0.0, 4062551.6163288243, 0.0, -0.2985821417389691]


In [6]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC)
        print(H)
        if H is not None:
            h_uav, w_uav = 490, 490
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            center_uav[0][0] = center_uav[0][0]-10# 形状为 (1, 2)
            center_uav[0][1] = center_uav[0][1]-45# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 328 image pairs.
UAV: uav/001.jpg - TIF: tif/131_856_1606.tif
(873, 2)
[[ 9.97021787e-01  4.92305065e-02 -6.38074112e+01]
 [-9.47032399e-02  1.09136469e+00  1.28450520e+02]
 [-1.35700130e-04  7.72145086e-05  1.00000000e+00]]
旋转角度 (度): -3.9426493428432194
无人机图像中心点在tif的位置：[183.35437123 329.8938374 ]
无人机图像中心点在地图上的位置：1039.354371230445,1935.8938373976048
无人机图像中心点的经纬度：34.24668132775793, 108.9020187226952
UAV: uav/002.jpg - TIF: tif/131_856_1606.tif
(882, 2)
[[ 1.02673924e+00  3.87662039e-02 -6.67052370e+01]
 [-6.13189491e-02  1.08007240e+00  1.17174836e+02]
 [-8.26148522e-05  5.10539061e-05  1.00000000e+00]]
旋转角度 (度): -2.719820219007671
无人机图像中心点在tif的位置：[184.02544192 321.74056958]
无人机图像中心点在地图上的位置：1040.02544192084,1927.7405695831985
无人机图像中心点的经纬度：34.246699404968666, 108.90202052264706
UAV: uav/003.jpg - TIF: tif/131_856_1606.tif
(934, 2)
[[ 1.03824274e+00  8.29758017e-02 -7.43565599e+01]
 [-7.00020255e-02  1.15893019e+00  9.42943465e+01]
 [-1.06354677e-04  1.63001731e-04  1.00000000e+00]]

/home/lty/anaconda3/envs/hloc/lib/python3.8/site-packages/osgeo/osr.py:410: FutureWarning: Neither osr.UseExceptions() nor osr.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


(671, 2)
[[ 1.07987941e+00 -3.13535539e-02 -4.78227251e+01]
 [-1.62827978e-03  9.73303492e-01  8.52723684e+01]
 [ 5.96431218e-05 -2.20403821e-04  1.00000000e+00]]
旋转角度 (度): 0.8294505940487732
无人机图像中心点在tif的位置：[205.8675565 288.2155141]
无人机图像中心点在地图上的位置：1061.8675565020574,1444.2155141020532
无人机图像中心点的经纬度：34.24777145704783, 108.90207910776368
UAV: uav/038.jpg - TIF: tif/89_856_1156.tif
(505, 2)
[[ 1.06039943e+00 -4.90687536e-02 -5.58780780e+01]
 [-7.06949846e-02  1.02508080e+00  5.55645335e+01]
 [-7.76611752e-05 -2.13435648e-04  1.00000000e+00]]
旋转角度 (度): -0.5941304739666164
无人机图像中心点在tif的位置：[195.40982471 259.79886627]
无人机图像中心点在地图上的位置：1051.409824706964,1415.798866266034
无人机图像中心点的经纬度：34.247834460858314, 108.9020510579412
UAV: uav/039.jpg - TIF: tif/89_856_1156.tif
(613, 2)
[[ 1.12314066e+00 -2.40524386e-02 -6.48338717e+01]
 [-5.03258459e-02  1.09741925e+00  3.38498593e+01]
 [-4.75598665e-05 -1.02637057e-04  1.00000000e+00]]
旋转角度 (度): -0.677885384277973
无人机图像中心点在tif的位置：[200.65526655 249.4145709

In [7]:
# from my_pkg.tools import extract_rotation_angle, rotate_point_z
# with open(loc_path, 'w') as loc_file:
#     i = 0
#     for img_uav, img_tif in pairs:
#         if i == 0:
#             print(f"UAV: {img_uav} - TIF: {img_tif}")
#             kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
#             matches,scores = get_matches(matches_path, img_uav, img_tif)
#             print(matches.shape)
#             pts1 = kp1[matches[:,0]]
#             pts2 = kp2[matches[:,1]]
#             H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC)
#             #rotation from H
#             theta = np.arctan2(H[1,0], H[0,0])
#             print(f"theta: {theta}")
#             print(H)
#             i+=1
# import numpy as np
# import cv2
# 
# # 示例 H 矩阵（请替换为你的实际 H 矩阵）
# # H = np.array([
# #     [1.2, 0.3, 100],
# #     [0.1, 1.1, 50],
# #     [0.001, 0.002, 1]
# # ])
# 
# angle = extract_rotation_angle(H)
# print(f"旋转角度 (度): {angle:.2f}")
# 
# point1 = np.array([-0.53, 160.55, 0])
# rotated_point1 = rotate_point_z(point1, -3.5)
# print(f"旋转前的点: {point1}")
# print(f"旋转后的点: {rotated_point1}")

# 生成地图轨迹

In [6]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory, plot_traj_tif
points_traj = []
with open("/home/lty/outputs/RealUAV/city2/loc.txt", 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3])   # 调整横坐标
        y_in_map = float(parts[4])  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

plot_traj_tif(
    map_image_path="/home/lty/outputs/RealUAV/city2/gt.png",
    loc_file_path=loc_path,
    output_image_path=output_dir/"traj_map.png",
    scale_factor=1,
)

# 绘制关键帧的单独景象匹配结果
map = cv2.imread("/home/lty/outputs/RealUAV/city2/gt.png", cv2.IMREAD_COLOR)
keyframe_mapping = parse_keyframe_file("/home/lty/code/ORB_SLAM3_detailed_comments/KeyFrameId.txt")
map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
cv2.imwrite(output_dir/"traj_KF.png", map_with_traj)

uav/001.jpg: 1039.35437123, 1935.8938374
uav/002.jpg: 1040.02544192, 1927.74056958
uav/003.jpg: 1040.8197203, 1913.29233598
uav/004.jpg: 1041.73634679, 1896.58253203
uav/005.jpg: 1041.76798015, 1883.2255045
uav/006.jpg: 1042.93756793, 1871.77980922
uav/007.jpg: 1044.05397885, 1859.49377705
uav/008.jpg: 1045.11207362, 1846.38017032
uav/009.jpg: 1043.93169885, 1831.50753816
uav/010.jpg: 1045.40448065, 1818.53281275
uav/011.jpg: 1044.84063939, 1804.58557452
uav/012.jpg: 1044.22905607, 1788.99505184
uav/013.jpg: 1043.57350487, 1774.84176274
uav/014.jpg: 1042.41011951, 1758.77612343
uav/015.jpg: 1046.34304046, 1747.02035146
uav/016.jpg: 1043.71502725, 1733.78830566
uav/017.jpg: 1043.82083126, 1719.06594644
uav/018.jpg: 1046.80750143, 1707.65926628
uav/019.jpg: 1043.01891269, 1688.54580787
uav/020.jpg: 1031.48559554, 1664.2460067
uav/021.jpg: 1036.07101379, 1652.36932647
uav/022.jpg: 1039.59644249, 1642.52903392
uav/023.jpg: 1042.97061525, 1635.68747984
uav/024.jpg: 1029.08071846, 1606.64653

FileNotFoundError: [Errno 2] No such file or directory: '/home/lty/code/ORB_SLAM3_detailed_comments/KeyFrameId.txt'

slam traj

In [7]:
#fusion traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj, draw_fusion_keyframe_traj
slam_traj_path = "/home/lty/outputs/RealUAV/city2/geoKFrame(step).txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"traj_KF.png", cv2.IMREAD_COLOR)
map_with_traj = draw_fusion_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"KF_slam_scenematch(step).png", map_with_traj)

True

In [7]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/outputs/RealUAV/city2/geoKFrame(slam).txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"KF_slam_scenematch.png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare.png", map_with_traj)

True

In [ ]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")